In [21]:
ratings = {
    "Tarik": [5, 4, 1, 0, 0],
    "Ali":   [5, 4, 2, 5, 4],
    "Sarah": [1, 2, 5, 1, 0],
    "John":  [4, 5, 1, 4, 5]
}

movies = [
    "Dark",
    "Interstellar",
    "Friends",
    "Dune",
    "Inception"
]

In [2]:
def cosine_similarity(a,b):
    dot = 0
    norm_a = 0
    norm_b = 0

    for i in range(len(a)):
        dot+= a[i]*b[i]
        norm_a += a[i]** 2
        norm_b += b[i]** 2

    norm_a = norm_a ** 0.5
    norm_b = norm_b ** 0.5

    return dot / (norm_a*norm_b)

In [5]:
def common_ratings(a,b):
    a_common = []
    b_common = []

    for i in range(len(a)):
        if a[i] != 0 and b[i] != 0:
            a_common.append(a[i])
            b_common.append(b[i])

    return a_common, b_common

In [7]:
a, b = common_ratings(
    ratings["Tarik"],
    ratings["Ali"]
)

print("Tarik common:", a)
print("Ali common:", b)
print(cosine_similarity(a,b))

Tarik common: [5, 4]
Ali common: [5, 5]
0.9938837346736188


In [9]:
target_user = "Tarik"

similarities = {}

for user in ratings:
    if user != target_user:

        a,b = common_ratings(
            ratings[target_user],
            ratings[user]
        )

        if len(a) == 0:
            similarity = 0
        else:
            similarity = cosine_similarity(a,b)
            
        similarities[user] = similarity
        
    print(similarities)

{}
{'Ali': 0.9938837346736188}
{'Ali': 0.9938837346736188, 'Sarah': 0.9938837346736188}


In [10]:
# We see that even though Sarah gave 1 for the rating of the first movie;
# Sarah and Ali has almost the same like pattern, to fix this we use mean centered values

In [12]:
def mean_center(values):
    mean = sum(values) / len(values)
    return [x - mean for x in values]

In [16]:
def user_similarity(a,b):
    a_common, b_common = common_ratings(a,b)

    if len(a_common) < 2:
        return 0

    a_centered = mean_center(a_common)
    b_centered = mean_center(b_common)

    norm_a = sum(x ** 2 for x in a_centered) ** 0.5
    norm_b = sum(x ** 2 for x in b_centered) ** 0.5

    if norm_a == 0 or norm_b == 0:
        return 0

    return cosine_similarity(a_centered, b_centered)

In [22]:
target_user = "Tarik"

similarities = {}

for user in ratings:
    if user != target_user:
        similarities[user] = user_similarity(
            ratings[target_user],
            ratings[user]
        )
print(similarities)

{'Ali': 0.9958705948858225, 'Sarah': -1.0, 'John': 0.8846153846153846}


In [24]:
target_user = "Tarik"

scores = {}
similarity_sums = {}

for user, similarity in similarities.items():

    if similarity <= 0:
        continue

    for i in range(len(movies)):

        # Only recommending movies Tarik has not rated
        if ratings[target_user][i] == 0 and ratings[user][i] != 0:

            movie = movies[i]

            if movie not in scores:

                scores[movie] = 0
                similarity_sums[movie] = 0

            scores[movie] += similarity *  ratings[user][i]
            similarity_sums[movie] += similarity

In [25]:
print(scores)
print(similarity_sums)

{'Dune': 8.517814512890652, 'Inception': 8.406559302620213}
{'Dune': 1.880485979501207, 'Inception': 1.880485979501207}


In [26]:
final_scores = {}

for movie in scores:
    final_scores[movie] = scores[movie] / similarity_sums[movie]

print(final_scores)

{'Dune': 4.529581504856513, 'Inception': 4.4704184951434875}


In [27]:
recommendations = sorted(
    final_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print(recommendations)

[('Dune', 4.529581504856513), ('Inception', 4.4704184951434875)]
